# 05 - ML Models

This notebook trains machine-learning baselines for Rossmann sales prediction.
Focus: robust preprocessing, time-aware validation, and interpretable model comparison.


## ML Workflow
1. Load engineered features from notebook 03.
2. Build chronological train/validation split.
3. Train baseline + tree models.
4. Evaluate RMSE, MAE, RMSPE.
5. Generate feature-importance diagnostics and test predictions.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 160)


In [ ]:
PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROCESSED_DIR / "train_features.csv"
test_path = PROCESSED_DIR / "test_features.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError("Run notebook 03 first to create train_features.csv and test_features.csv")

train_df = pd.read_csv(train_path, parse_dates=["Date"])
test_df = pd.read_csv(test_path, parse_dates=["Date"])

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)


## Target, Features, and Split
- Predict `Sales` on original scale.
- Use chronological split for realistic forecasting evaluation.


In [ ]:
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.sqrt(np.mean(np.square((y_true[mask] - y_pred[mask]) / y_true[mask])))

# Drop leakage-prone or unavailable-at-inference columns.
exclude_cols = {"Sales", "Customers", "log_sales", "Id"}
features = [c for c in train_df.columns if c not in exclude_cols]

cutoff = train_df["Date"].quantile(0.85)
tr = train_df[train_df["Date"] <= cutoff].copy()
va = train_df[train_df["Date"] > cutoff].copy()

X_train = tr[features].copy()
y_train = tr["Sales"].copy()
X_valid = va[features].copy()
y_valid = va["Sales"].copy()

print("Cutoff:", cutoff)
print("Train rows:", len(X_train), "Valid rows:", len(X_valid))


In [ ]:
# Convert date to ordinal day index for model-friendly numeric signal.
for df_ in [X_train, X_valid]:
    df_["DateOrdinal"] = pd.to_datetime(df_["Date"]).map(pd.Timestamp.toordinal)
    df_.drop(columns=["Date"], inplace=True)

X_test = test_df[[c for c in features if c in test_df.columns]].copy()
X_test["DateOrdinal"] = pd.to_datetime(X_test["Date"]).map(pd.Timestamp.toordinal)
X_test.drop(columns=["Date"], inplace=True)


## Preprocessing Pipeline
- Numeric: median imputation.
- Categorical: one-hot encoding with unknown-category handling.


In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

print("Numeric cols:", len(num_cols), "Categorical cols:", len(cat_cols))


## Train Baseline Models


In [ ]:
models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=220,
        max_depth=18,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
    ),
    "HistGBR": HistGradientBoostingRegressor(
        learning_rate=0.06,
        max_depth=10,
        max_iter=350,
        random_state=42,
    ),
}

results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    pred = np.clip(pipe.predict(X_valid), a_min=0, a_max=None)

    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    mae = mean_absolute_error(y_valid, pred)
    r = rmspe(y_valid.values, pred)

    results.append({"model": name, "RMSE": rmse, "MAE": mae, "RMSPE": r})
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSPE")
results_df


## Optional XGBoost Benchmark
This cell runs only if `xgboost` is installed in your environment.


In [ ]:
try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

    xgb_pipe = Pipeline([
        ("prep", preprocessor),
        ("model", xgb),
    ])
    xgb_pipe.fit(X_train, y_train)
    xgb_pred = np.clip(xgb_pipe.predict(X_valid), a_min=0, a_max=None)

    xgb_row = {
        "model": "XGBoost",
        "RMSE": np.sqrt(mean_squared_error(y_valid, xgb_pred)),
        "MAE": mean_absolute_error(y_valid, xgb_pred),
        "RMSPE": rmspe(y_valid.values, xgb_pred),
    }
    results_df = pd.concat([results_df, pd.DataFrame([xgb_row])], ignore_index=True).sort_values("RMSPE")
    fitted["XGBoost"] = xgb_pipe
except Exception as e:
    print("XGBoost skipped:", e)

results_df


## Feature Importance (Best Tree Model)


In [ ]:
# Pick the best available tree model from fitted models.
best_name = results_df.iloc[0]["model"]
print("Best model by RMSPE:", best_name)

if best_name in {"RandomForest", "XGBoost"}:
    best_pipe = fitted[best_name]
    model = best_pipe.named_steps["model"]
    prep = best_pipe.named_steps["prep"]

    feature_names = prep.get_feature_names_out()
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    top_imp = importance.head(20)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_imp, y="feature", x="importance")
    plt.title(f"Top 20 Feature Importances - {best_name}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ml_top_feature_importance.png", dpi=140)
    plt.show()

    top_imp
else:
    print("Tree-based feature importance not available for", best_name)


## Generate Kaggle-Format Predictions


In [ ]:
best_name = results_df.iloc[0]["model"]
best_pipe = fitted[best_name]

test_pred = np.clip(best_pipe.predict(X_test), a_min=0, a_max=None)

if "Id" in test_df.columns:
    sub = pd.DataFrame({"Id": test_df["Id"], "Sales": test_pred})
else:
    # Fallback if Id is unavailable in some intermediate table.
    sub = pd.DataFrame({"Sales": test_pred})

sub_path = PROCESSED_DIR / "submission_baseline.csv"
sub.to_csv(sub_path, index=False)
print("Saved:", sub_path)
sub.head()


## Next Steps
1. Tune best model with walk-forward validation and early stopping.
2. Add richer store-level lag features (28/56 day seasonality windows).
3. Compare final ML model against SARIMAX benchmark from notebook 04.
